In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense
from keras.models import Sequential

In [3]:
from pathlib import Path

dataset_root = Path("/kaggle/input/datasets/anikasadiaopt/thyroid-nodule/TN5000_forReview")

image_dir = dataset_root / "JPEGImages/"
annotation_dir = dataset_root / "Annotations/"
img_files = sorted(image_dir.glob("*.jpg"))

In [4]:
import cv2
from google.colab.patches import cv2_imshow


img = cv2.imread(str(img_files[10]))
img.shape

print(f"Total images: {len(img_files)}")

Total images: 5000


In [5]:
print("Height:", img.shape[0])
print("Width :", img.shape[1])
print("Channels:", img.shape[2])
print("Data type:", img.dtype)
print("File size (KB):", round(image_dir.stat().st_size / 1024, 2))

Height: 628
Width : 818
Channels: 3
Data type: uint8
File size (KB): 0.0


In [6]:
from PIL import Image
import xml.etree.ElementTree as ET
from collections import Counter

sizes = []
counter = Counter()

for img_path in img_files:
    xml_path = annotation_dir / f"{img_path.stem}.xml"

    # Store image size
    with Image.open(img_path) as img:
        sizes.append(img.size)

    # Read corresponding annotation
    if xml_path.exists():
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Count every <object>'s <name>
        for obj in root.findall("object"):
            name = obj.find("name").text.strip()
            counter[name] += 1

# Print counts
print("Class distribution:")
for cls in sorted(counter.keys()):
    print(f"Class {cls}: {counter[cls]}")

size_counts = Counter(sizes)
for size, count in size_counts.items():
  print(f"{size}: {count} images")

# Print image size information
print("\nNumber of images:", len(sizes))
print("Unique image sizes:", set(sizes))
print("Some sizes", list(set(sizes))[:10])

Class distribution:
Class 0: 1426
Class 1: 3574
(718, 500): 3975 images
(818, 628): 649 images
(498, 429): 205 images
(439, 368): 124 images
(919, 646): 24 images
(492, 391): 3 images
(501, 458): 9 images
(466, 349): 6 images
(677, 432): 5 images

Number of images: 5000
Unique image sizes: {(919, 646), (498, 429), (677, 432), (492, 391), (501, 458), (466, 349), (439, 368), (718, 500), (818, 628)}
Some sizes [(919, 646), (498, 429), (677, 432), (492, 391), (501, 458), (466, 349), (439, 368), (718, 500), (818, 628)]


In [7]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:00


In [8]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.100 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 6998.9/8062.4 GB disk)


In [9]:
from ultralytics import YOLO

# Load the pretrained model
model = YOLO("yolo11m.pt") 

# Start training
model.train(
    data="/kaggle/input/datasets/anikasadiaopt/data-for-yolo/data (1).yaml", 
    epochs=200, 
    imgsz=800, 
    batch=16, 
    workers=2, 
    device=0,          
    patience=25, 
    optimizer='SGD', 
    scale=0.5,
    lr0=0.001,      
    multi_scale=True, 
    box = 8,
    dfl = 2,
    cls = 0.5,
    iou = 0.7, 
    project="/kaggle/working/YOLO_for_detection/YOLO_Thyroid_Nodule_Detection", 
    name="YOLO11m"
)


Ultralytics 8.4.100 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/anikasadiaopt/data-for-yolo/data (1).yaml, degrees=0.0, deterministic=True, device=0, dfl=2, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=YOLO11m, nbs=64, nms=False, opset=None, optimize=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bbabda72720>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

***fetching weights***

# Classification using CNN


In [10]:
from pathlib import Path
import xml.etree.ElementTree as ET

def load_split(split_dir, annotation_dir):
    split_dir = Path(split_dir)

    imgs = sorted(split_dir.glob("*.png"))
    labels = []

    for img in imgs:
        xml_path = annotation_dir / f"{img.stem}.xml"

        tree = ET.parse(xml_path)
        root = tree.getroot()

        label = int(root.find("object").find("name").text.strip())
        labels.append(label)

    return imgs, labels

In [11]:
from pathlib import Path
roi_root = Path("/kaggle/input/datasets/anikasadiaopt/roi-dataset/ROI_Dataset/ROI_Images")

train_images, train_labels = load_split(roi_root / "train", annotation_dir)
val_images, val_labels = load_split(roi_root / "val", annotation_dir)
test_images, test_labels = load_split(roi_root / "test", annotation_dir)

print(f"Train: {len(train_images)} images, {len(train_labels)} labels")
print(f"Validation: {len(val_images)} images, {len(val_labels)} labels")
print(f"Test: {len(test_images)} images, {len(test_labels)} labels")

Train: 4000 images, 4000 labels
Validation: 500 images, 500 labels
Test: 500 images, 500 labels


In [12]:
!pip install albumentations

In [13]:
from PIL import Image
import xml.etree.ElementTree as ET
from collections import Counter

sizes = []
counter = Counter()

roi_widths = []
roi_heights = []
roi_areas = []

for img_path in img_files :
    xml_path = annotation_dir / f"{img_path.stem}.xml"

    # Image size
    with Image.open(img_path) as img:
        img_w, img_h = img.size
        sizes.append((img_w, img_h))

    if xml_path.exists():
        tree = ET.parse(xml_path)
        root = tree.getroot()

        for obj in root.findall("object"):

            # Class
            name = obj.find("name").text.strip()
            counter[name] += 1

            # Bounding box
            box = obj.find("bndbox")

            xmin = int(box.find("xmin").text)
            ymin = int(box.find("ymin").text)
            xmax = int(box.find("xmax").text)
            ymax = int(box.find("ymax").text)

            roi_w = xmax - xmin
            roi_h = ymax - ymin
            roi_area = roi_w * roi_h

            roi_widths.append(roi_w)
            roi_heights.append(roi_h)
            roi_areas.append(roi_area)

# -----------------------------
# Results
# -----------------------------

print("Class distribution:")
for cls in sorted(counter.keys()):
    print(f"Class {cls}: {counter[cls]}")

size_counts = Counter(sizes)

print("\nImage sizes:")
for size, count in size_counts.items():
    print(f"{size}: {count} images")

print("\nROI Statistics")
print(f"Total ROIs: {len(roi_widths)}")

print(f"ROI Width  : Min={min(roi_widths)}, Max={max(roi_widths)}, Avg={sum(roi_widths)/len(roi_widths):.2f}")

print(f"ROI Height : Min={min(roi_heights)}, Max={max(roi_heights)}, Avg={sum(roi_heights)/len(roi_heights):.2f}")

print(f"ROI Area   : Min={min(roi_areas)}, Max={max(roi_areas)}, Avg={sum(roi_areas)/len(roi_areas):.2f}")

Class distribution:
Class 0: 1426
Class 1: 3574

Image sizes:
(718, 500): 3975 images
(818, 628): 649 images
(498, 429): 205 images
(439, 368): 124 images
(919, 646): 24 images
(492, 391): 3 images
(501, 458): 9 images
(466, 349): 6 images
(677, 432): 5 images

ROI Statistics
Total ROIs: 5000
ROI Width  : Min=17, Max=717, Avg=146.65
ROI Height : Min=18, Max=476, Avg=119.32
ROI Area   : Min=459, Max=287517, Avg=23016.58


**CNN**

In [14]:
len(train_images)

4000

In [15]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])
test_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

In [16]:
import sys

sys.path.insert(0, "/kaggle/input/datasets/anikasadiaopt/thyroiddataset")

from Thyroid_Dataset import Thyroid_Dataset

print(Thyroid_Dataset.__module__)

from Thyroid_Dataset import Thyroid_Dataset

train_dataset = Thyroid_Dataset(
    img_path=train_images,
    annotation_dir=annotation_dir,
    transform=train_transform
)

val_dataset = Thyroid_Dataset(
    img_path=val_images,
    annotation_dir=annotation_dir,
    transform=val_transform
)

test_dataset = Thyroid_Dataset(
    img_path=test_images,
    annotation_dir=annotation_dir,
    transform=val_transform
)

Thyroid_Dataset


In [17]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [18]:
import torch.nn as nn

class ThyroidCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [19]:
model = ThyroidCNN()

In [20]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [21]:
import torch

# Optimized configuration to prevent overfitting
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=3e-4,               # Reduced learning rate gives model more time to regularize
    weight_decay=1e-3     # Significantly increased weight decay to penalize large weights
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=30, 
)


In [22]:
import os

save_dir = "Output/model/Classification/"
os.makedirs(save_dir, exist_ok=True)

In [23]:
num_epochs = 100
best_val_acc = 0
patience = 20
counter = 0

for epoch in range(num_epochs):

    model.train()

    running_loss = 0

    correct = 0

    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    scheduler.step()

        
    train_acc = correct / total

    #validation
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _,predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = val_correct / val_total
    val_loss = val_loss / len(val_loader)

    if val_acc >best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(),"Output/model/Classification/thyroid_cnn.pt")
        print("model saved.")
        counter = 0
    else: 
        counter +=1
        

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {running_loss/len(train_loader):.4f} | "
        f"Train Acc: {train_acc:.2f} |"
        f"Val Loss: {val_loss:.2f} |"
        f"Val Acc: {val_acc:.2f}"
    )
    if counter> patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        break
        
    

model saved.
Epoch 01 | Train Loss: 0.5707 | Train Acc: 0.74 |Val Loss: 0.50 |Val Acc: 0.80
Epoch 02 | Train Loss: 0.5260 | Train Acc: 0.79 |Val Loss: 0.52 |Val Acc: 0.78
model saved.
Epoch 03 | Train Loss: 0.5097 | Train Acc: 0.80 |Val Loss: 0.49 |Val Acc: 0.81
model saved.
Epoch 04 | Train Loss: 0.5075 | Train Acc: 0.80 |Val Loss: 0.46 |Val Acc: 0.82
Epoch 05 | Train Loss: 0.4998 | Train Acc: 0.80 |Val Loss: 0.46 |Val Acc: 0.82
Epoch 06 | Train Loss: 0.4986 | Train Acc: 0.80 |Val Loss: 0.46 |Val Acc: 0.82
Epoch 07 | Train Loss: 0.4923 | Train Acc: 0.81 |Val Loss: 0.49 |Val Acc: 0.82
model saved.
Epoch 08 | Train Loss: 0.4885 | Train Acc: 0.81 |Val Loss: 0.44 |Val Acc: 0.84
Epoch 09 | Train Loss: 0.4851 | Train Acc: 0.81 |Val Loss: 0.52 |Val Acc: 0.82
Epoch 10 | Train Loss: 0.4815 | Train Acc: 0.82 |Val Loss: 0.46 |Val Acc: 0.83
model saved.
Epoch 11 | Train Loss: 0.4785 | Train Acc: 0.82 |Val Loss: 0.45 |Val Acc: 0.85
Epoch 12 | Train Loss: 0.4725 | Train Acc: 0.82 |Val Loss: 0.44 |V

In [24]:
import torch
import torch.nn.functional as F

model.load_state_dict(torch.load("Output/model/Classification/thyroid_cnn.pt"))
model.eval()

test_loss = 0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Original image prediction
        outputs_original = model(images)

        # Horizontal flip prediction
        images_flip = torch.flip(images, dims=[3])   # Flip along width
        outputs_flip = model(images_flip)

        # Convert logits to probabilities
        probs_original = F.softmax(outputs_original, dim=1)
        probs_flip = F.softmax(outputs_flip, dim=1)

        # Average probabilities
        probs = (probs_original + probs_flip) / 2

        # Compute loss using original outputs
        loss = criterion(outputs_original, labels)
        test_loss += loss.item()

        # Final prediction
        _, predicted = probs.max(1)

        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_loss /= len(test_loader)
test_acc = test_correct / test_total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}")

Test Loss: 0.4232
Test Accuracy: 0.84


GRAD-CAM

Loaded CNN for GRAD-CAM

In [25]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create your custom model
model = ThyroidCNN().to(device)

# Load trained weights (only if they were saved from ThyroidCNN)
model.load_state_dict(
    torch.load("/kaggle/working/Output/model/Classification/thyroid_cnn.pt", map_location=device)
)

model.eval()

ThyroidCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True

In [26]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 66.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44285 sha256=e5d247a68520f848756a112b8b79d129130d928a2b2a3aba4693f34ac2026383
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [27]:
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [28]:
from pytorch_grad_cam import GradCAM

target_layers = [model.features[12]]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

In [29]:
output_dir = Path("Output/GradCAM_Output")
output_dir.mkdir(exist_ok=True)

In [30]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])


In [31]:
test_dir = Path("/kaggle/input/datasets/anikasadiaopt/roi-dataset/ROI_Dataset/ROI_Images/test")
image_paths = sorted(test_dir.glob("*.png"))

In [32]:
import torch
from pytorch_grad_cam.utils.image import show_cam_on_image
import cv2
for img_path in image_paths:

    # Read image
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    rgb_img = img.astype(np.float32) / 255.0

    # CNN preprocessing
    input_tensor = transform(image=img)["image"].unsqueeze(0).to(device)

    # Prediction
    with torch.no_grad():
        output = model(input_tensor)

    pred = output.argmax(1).item()

    conf = torch.softmax(output, dim=1)[0, pred].item()

    # Grad-CAM
    grayscale_cam = cam(input_tensor=input_tensor)[0]

    # Overlay
    visualization = show_cam_on_image(
        rgb_img,
        grayscale_cam,
        use_rgb=True
    )

    # Save
    save_name = output_dir / f"{img_path.stem}_gradcam.png"

    cv2.imwrite(
        str(save_name),
        cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR)
    )

    print(f"{img_path.stem} -> Class:{pred}  Confidence:{conf:.3f}")

000009 -> Class:1  Confidence:0.574
000018 -> Class:0  Confidence:0.886
000030 -> Class:0  Confidence:0.888
000044 -> Class:0  Confidence:0.815
000046 -> Class:1  Confidence:0.720
000062 -> Class:0  Confidence:0.613
000066 -> Class:1  Confidence:0.957
000069 -> Class:1  Confidence:0.535
000070 -> Class:1  Confidence:0.858
000091 -> Class:1  Confidence:0.918
000094 -> Class:1  Confidence:0.857
000097 -> Class:0  Confidence:0.791
000107 -> Class:1  Confidence:0.767
000110 -> Class:1  Confidence:0.835
000133 -> Class:1  Confidence:0.738
000135 -> Class:1  Confidence:0.850
000140 -> Class:1  Confidence:0.586
000152 -> Class:0  Confidence:0.695
000158 -> Class:1  Confidence:0.813
000180 -> Class:1  Confidence:0.881
000185 -> Class:1  Confidence:0.658
000204 -> Class:0  Confidence:0.858
000215 -> Class:0  Confidence:0.783
000220 -> Class:0  Confidence:0.650
000222 -> Class:0  Confidence:0.534
000228 -> Class:0  Confidence:0.724
000229 -> Class:1  Confidence:0.668
000231 -> Class:0  Confidenc